In [60]:
'''!pip install -q pandas numpy yfinance spacy nltk underthesea scikit-learn \
    pdfminer.six wordcloud matplotlib pyLDAvis chatterbot lxml tensorflow
!python -m spacy download en_core_web_sm -q'''

'!pip install -q pandas numpy yfinance spacy nltk underthesea scikit-learn     pdfminer.six wordcloud matplotlib pyLDAvis chatterbot lxml tensorflow\n!python -m spacy download en_core_web_sm -q'

In [61]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = "/content/drive/MyDrive/NLP_Finance"
os.makedirs(ROOT, exist_ok=True)
os.chdir(ROOT)

for d in ["core", "utils", "models", "models/saved",
          "data/raw", "data/processed", "data/lexicons"]:
    os.makedirs(d, exist_ok=True)

open("core/__init__.py", "w").close()
open("models/__init__.py", "w").close()

print("Thư mục hiện tại:", os.getcwd())
print(os.listdir("."))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục hiện tại: /content/drive/MyDrive/NLP_Finance
['core', 'utils', 'data', 'models', 'external_corpus', 'results', 'requirements.txt', 'main.py', 'preprocess_vietnamese_news.py', 'caoData.ipynb', 'db.sqlite3', 'Test.ipynb', 'writeFile.ipynb']


In [62]:
%%writefile requirements.txt
pandas
numpy
yfinance
spacy
nltk
underthesea
scikit-learn
pdfminer.six
wordcloud
matplotlib
pyLDAvis
chatterbot
lxml
tensorflow
vnstock

Overwriting requirements.txt


In [63]:
%%writefile models/lda_topic_model.py
"""
LDA topic modeling wrapper.
Bám theo Case Study 3 "Document Summarization" trong sách.
Hỗ trợ cả tiếng Anh (stop_words='english') và tiếng Việt (stop_words=None,
vì văn bản tiếng Việt đã được xóa stopwords ở bước tiền xử lý).
"""

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


class LDATopicModel:
    def __init__(self, n_topics=5, stop_words='english', ngram_range=(1, 1)):
        self.n_topics = n_topics
        self.vectorizer = CountVectorizer(ngram_range=ngram_range, stop_words=stop_words)
        self.lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
        self.dtm = None
        self.feature_names = None

    def fit(self, documents: list):
        self.dtm = self.vectorizer.fit_transform(documents)
        self.lda.fit(self.dtm)
        self.feature_names = np.array(self.vectorizer.get_feature_names_out())
        return self

    def transform(self, documents: list = None):
        dtm = self.vectorizer.transform(documents) if documents is not None else self.dtm
        return self.lda.transform(dtm)

    def print_top_words(self, n_words=10):
        sorting = np.argsort(self.lda.components_)[:, ::-1]
        topics = {}
        for topic_idx in range(self.n_topics):
            top_words = self.feature_names[sorting[topic_idx][:n_words]]
            topics[f"Topic {topic_idx + 1}"] = list(top_words)
            print(f"Topic {topic_idx + 1}: {list(top_words)}")
        return topics

    def visualize(self):
        try:
            import pyLDAvis
            import pyLDAvis.sklearn
            return pyLDAvis.sklearn.prepare(self.lda, self.dtm, self.vectorizer)
        except Exception as e:
            print(f"[WARN] Không thể tạo pyLDAvis: {e}")
            return None

    def assign_dominant_topic(self, documents: list):
        doc_topic_matrix = self.transform(documents)
        return np.argmax(doc_topic_matrix, axis=1) + 1


def run_lda_pipeline(documents: list, n_topics=5, n_words=10,
                      visualize=False, stop_words='english'):
    print(f">> [LDA] Đang huấn luyện mô hình với {n_topics} topics "
          f"(stop_words={stop_words})...")
    model = LDATopicModel(n_topics=n_topics, stop_words=stop_words)
    model.fit(documents)
    topics = model.print_top_words(n_words=n_words)
    panel = model.visualize() if visualize else None
    return model, topics, panel


if __name__ == "__main__":
    demo_docs = [
        "The stock price of google jumps on the earning data today",
        "Google plunge on China Data",
        "Apple reports record profit growth this quarter",
        "Amazon revenue increases due to strong holiday sales",
    ]
    run_lda_pipeline(demo_docs, n_topics=3, n_words=5)

Overwriting models/lda_topic_model.py


In [64]:
%%writefile models/lstm_sentiment.py
"""
LSTM-based sentiment classification model.
Bám theo Case Study 1, mục 4.2 trong sách.
"""

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense


class LSTMSentimentModel:
    def __init__(self, vocab_size=20000, max_len=50, embedding_dim=300):
        self.vocab_size = vocab_size
        self.max_len = max_len
        self.embedding_dim = embedding_dim
        self.tokenizer = Tokenizer(num_words=vocab_size)
        self.model = None
        self.history = None

    def prepare_sequences(self, texts, fit_tokenizer=True):
        if fit_tokenizer:
            self.tokenizer.fit_on_texts(texts)
        sequences = self.tokenizer.texts_to_sequences(texts)
        return pad_sequences(sequences, maxlen=self.max_len)

    def build_model(self):
        model = Sequential()
        model.add(Embedding(self.vocab_size, self.embedding_dim, input_length=self.max_len))
        model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
        self.model = model
        return model

    def train(self, headlines, labels, epochs=3, validation_split=0.4, batch_size=32, verbose=1):
        X = self.prepare_sequences(headlines, fit_tokenizer=True)
        y = np.array(labels)
        if self.model is None:
            self.build_model()
        self.history = self.model.fit(X, y, epochs=epochs, validation_split=validation_split,
                                       batch_size=batch_size, verbose=verbose)
        return self.history

    def predict(self, texts):
        X = self.prepare_sequences(texts, fit_tokenizer=False)
        probs = self.model.predict(X, verbose=0)
        return (probs > 0.5).astype(int).flatten(), probs.flatten()

    def evaluate(self, texts, true_labels):
        preds, probs = self.predict(texts)
        acc = accuracy_score(true_labels, preds)
        print(f"[LSTM] Accuracy: {acc:.4f}")
        print(classification_report(true_labels, preds))
        return acc

    def save(self, path="models/saved/lstm_sentiment.h5"):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.model.save(path)
        print(f"[INFO] Đã lưu model LSTM tại {path}")

    def load(self, path="models/saved/lstm_sentiment.h5"):
        self.model = load_model(path)
        print(f"[INFO] Đã load model LSTM từ {path}")


def run_lstm_pipeline(csv_path="data/raw/LabelledNewsData.csv",
                       text_col="headline", label_col="sentiment",
                       test_size=0.2, epochs=3):
    print(">> [LSTM] Đang load dữ liệu đã gán nhãn...")
    try:
        df = pd.read_csv(csv_path, encoding="ISO-8859-1")
    except FileNotFoundError:
        print(f"[WARN] Không tìm thấy {csv_path}. Dùng dữ liệu demo.")
        df = pd.DataFrame({
            'headline': [
                "Apple posts record profit growth this quarter",
                "Company faces bankruptcy after massive losses",
                "Stock jumps after positive earnings beat",
                "Shares plunge amid fraud investigation"
            ],
            'sentiment': [1, 0, 1, 0]
        })

    X_train, X_test, y_train, y_test = train_test_split(
        df[text_col], df[label_col], test_size=test_size, random_state=42
    )
    lstm = LSTMSentimentModel()
    lstm.train(X_train.tolist(), y_train.tolist(), epochs=epochs)
    lstm.evaluate(X_test.tolist(), y_test.tolist())
    lstm.save()
    return lstm


if __name__ == "__main__":
    run_lstm_pipeline()

Overwriting models/lstm_sentiment.py


In [65]:
%%writefile models/ner_model.py
"""
Custom Named Entity Recognition (NER) model bằng spaCy.
Bám theo Case Study 2, mục 4-5 "Chatbot Digital Assistant" trong sách.

Đã sửa lỗi overfitting của phiên bản trước:
1. Mở rộng templates (đa dạng cấu trúc câu, có nhiễu, có câu không entity)
2. Sửa lỗi sent.index() gán sai vị trí khi entity là chuỗi con
3. Tách train/dev theo tổ hợp company-ratio KHÔNG trùng nhau (đo generalization thật)
4. Thêm early stopping dựa trên dev loss
5. Giới hạn/loại bỏ câu trùng lặp
"""

import random
import re
import spacy
from spacy.util import minibatch, compounding
from spacy.training import Example


class FinancialNERModel:
    def __init__(self, labels=None):
        self.labels = labels or ['RATIO', 'COMPANY']
        self.nlp = spacy.blank("en")
        self.ner = self.nlp.add_pipe("ner")
        for label in self.labels:
            self.ner.add_label(label)

    # ------------------------------------------------------------
    # 1. Templates đa dạng hơn nhiều (khác cấu trúc, có nhiễu, có câu
    #    không chứa entity nào để model học "không phải câu nào cũng
    #    có RATIO/COMPANY")
    # ------------------------------------------------------------
    DEFAULT_TEMPLATES = [
        'Get me the {ratio} for {company}',
        'What is the {ratio} for {company}?',
        'Tell me the {ratio} for {company}',
        'Can you show the {ratio} of {company}?',
        'I want to know the {ratio} for {company}',
        'Please provide {company} {ratio}',
        'How is {company} doing in terms of {ratio}?',
        'Show me {company}\'s {ratio} please',
        '{company} - what is its {ratio}?',
        'Give me {ratio} data for {company} today',
        'Looking for the {ratio} of {company}, can you help?',
        'What about the {ratio} for {company} this quarter?',
    ]

    # Câu KHÔNG chứa entity nào - giúp model không "ảo giác" entity ở mọi câu
    NEGATIVE_SENTENCES = [
        "Hello, how are you today?",
        "Can you help me with something?",
        "What is the weather like?",
        "Thank you very much for your help",
        "I would like to schedule a meeting",
        "This is a great conversation",
        "Please tell me a joke",
        "What time is it right now?",
    ]

    @staticmethod
    def _find_all_occurrences(text, sub):
        """Tìm TẤT CẢ vị trí xuất hiện của sub trong text (dùng word boundary
        để tránh match nhầm chuỗi con, ví dụ 'PE' không match trong 'Pepsi')."""
        pattern = r'\b' + re.escape(sub) + r'\b'
        return [m.start() for m in re.finditer(pattern, text)]

    @classmethod
    def generate_training_sample(cls, string_templates, ratios_list, companies_list):
        """
        Sinh 1 câu training, đã sửa lỗi gán sai vị trí entity khi
        ratio/company là chuỗi con của từ khác trong câu.
        """
        template = random.choice(string_templates)
        ratio = random.choice(ratios_list)
        company = random.choice(companies_list)
        sent = template.format(ratio=ratio, company=company)

        ratio_positions = cls._find_all_occurrences(sent, ratio)
        company_positions = cls._find_all_occurrences(sent, company)

        if not ratio_positions or not company_positions:
            # Không tìm được vị trí an toàn (word boundary) -> bỏ qua mẫu này
            return None

        # Lấy vị trí đầu tiên tìm được với word boundary (an toàn hơn .index())
        ratio_start = ratio_positions[0]
        company_start = company_positions[0]

        entities = [
            (ratio_start, ratio_start + len(ratio), 'RATIO'),
            (company_start, company_start + len(company), 'COMPANY'),
        ]
        # Sắp xếp entity theo vị trí xuất hiện (spaCy yêu cầu thứ tự tăng dần)
        entities.sort(key=lambda x: x[0])

        return (sent, {"entities": entities})

    @classmethod
    def generate_negative_sample(cls):
        """Sinh câu không chứa entity nào (annotation rỗng)"""
        sent = random.choice(cls.NEGATIVE_SENTENCES)
        return (sent, {"entities": []})

    def generate_training_data(self, string_templates, ratios_list, companies_list,
                                n_samples=100, negative_ratio=0.15, dedup=True):
        """
        Sinh dữ liệu training với:
        - negative_ratio: tỷ lệ câu không chứa entity (mặc định 15%)
        - dedup: loại bỏ câu trùng lặp hoàn toàn để tránh model học thuộc
        """
        data = []
        seen_sentences = set()
        n_negative = int(n_samples * negative_ratio)
        n_positive = n_samples - n_negative

        attempts = 0
        max_attempts = n_positive * 5  # tránh vòng lặp vô hạn nếu combo gần hết
        while len(data) < n_positive and attempts < max_attempts:
            attempts += 1
            sample = self.generate_training_sample(string_templates, ratios_list, companies_list)
            if sample is None:
                continue
            sent, ents = sample
            if dedup and sent in seen_sentences:
                continue
            seen_sentences.add(sent)
            data.append(sample)

        for _ in range(n_negative):
            data.append(self.generate_negative_sample())

        random.shuffle(data)
        print(f"[INFO] Đã sinh {len(data)} câu training "
              f"({n_positive} có entity, {n_negative} không có entity, "
              f"{len(seen_sentences)} câu duy nhất).")
        return data

    def generate_train_dev_split(self, string_templates, ratios_list, companies_list,
                                  n_train=150, n_dev=40, negative_ratio=0.15):
        """
        Tách train/dev theo tổ hợp company-ratio KHÔNG TRÙNG NHAU,
        để đo được model có generalize thật hay chỉ học thuộc.
        """
        all_combos = [(r, c) for r in ratios_list for c in companies_list]
        random.shuffle(all_combos)

        n_dev_combos = max(1, len(all_combos) // 5)  # ~20% combo dành cho dev
        dev_combos = set(all_combos[:n_dev_combos])
        train_combos = set(all_combos[n_dev_combos:])

        def _sample_from_combos(combos, n_samples, negative_ratio):
            data = []
            seen = set()
            n_negative = int(n_samples * negative_ratio)
            n_positive = n_samples - n_negative
            attempts, max_attempts = 0, n_positive * 8

            combos_list = list(combos)
            while len(data) < n_positive and attempts < max_attempts and combos_list:
                attempts += 1
                ratio, company = random.choice(combos_list)
                template = random.choice(string_templates)
                sent = template.format(ratio=ratio, company=company)

                ratio_pos = self._find_all_occurrences(sent, ratio)
                company_pos = self._find_all_occurrences(sent, company)
                if not ratio_pos or not company_pos:
                    continue
                if sent in seen:
                    continue
                seen.add(sent)

                ents = sorted([
                    (ratio_pos[0], ratio_pos[0] + len(ratio), 'RATIO'),
                    (company_pos[0], company_pos[0] + len(company), 'COMPANY'),
                ], key=lambda x: x[0])
                data.append((sent, {"entities": ents}))

            for _ in range(n_negative):
                data.append(self.generate_negative_sample())

            random.shuffle(data)
            return data

        train_data = _sample_from_combos(train_combos, n_train, negative_ratio)
        dev_data = _sample_from_combos(dev_combos, n_dev, negative_ratio)

        print(f"[INFO] Train: {len(train_data)} câu (từ {len(train_combos)} tổ hợp company-ratio)")
        print(f"[INFO] Dev  : {len(dev_data)} câu (từ {len(dev_combos)} tổ hợp company-ratio "
              f"KHÔNG xuất hiện trong train)")
        return train_data, dev_data

    # ------------------------------------------------------------
    # 2. Huấn luyện với early stopping dựa trên dev loss
    # ------------------------------------------------------------
    def train(self, train_data, dev_data=None, n_iter=30, drop=0.35,
              patience=5, min_delta=0.001):
        """
        patience: số vòng lặp cho phép dev loss không cải thiện trước khi dừng sớm
        min_delta: mức cải thiện tối thiểu để tính là "có tiến bộ"
        """
        optimizer = self.nlp.begin_training()
        other_pipes = [p for p in self.nlp.pipe_names if p != "ner"]

        best_dev_loss = float('inf')
        patience_counter = 0
        best_model_bytes = None

        with self.nlp.disable_pipes(*other_pipes):
            sizes = compounding(1.0, 4.0, 1.001)
            for itn in range(n_iter):
                random.shuffle(train_data)
                batches = minibatch(train_data, size=sizes)
                losses = {}
                for batch in batches:
                    examples = []
                    for text, annotations in batch:
                        doc = self.nlp.make_doc(text)
                        examples.append(Example.from_dict(doc, annotations))
                    self.nlp.update(examples, sgd=optimizer, drop=drop, losses=losses)

                train_loss = losses.get('ner', 0.0)

                if dev_data:
                    dev_loss = self._evaluate_loss(dev_data)
                    print(f"  Iter {itn + 1}/{n_iter} - Train loss: {train_loss:.4f} "
                          f"- Dev loss: {dev_loss:.4f}")

                    if dev_loss < best_dev_loss - min_delta:
                        best_dev_loss = dev_loss
                        patience_counter = 0
                        best_model_bytes = self.nlp.to_bytes()
                    else:
                        patience_counter += 1
                        if patience_counter >= patience:
                            print(f"[INFO] Early stopping tại iter {itn + 1} "
                                  f"(dev loss không cải thiện sau {patience} vòng)")
                            break
                else:
                    if (itn + 1) % 5 == 0 or itn == n_iter - 1:
                        print(f"  Iter {itn + 1}/{n_iter} - Train loss: {train_loss:.4f}")

        # Khôi phục lại model tốt nhất theo dev loss (nếu có dùng dev_data)
        if best_model_bytes is not None:
            self.nlp.from_bytes(best_model_bytes)
            print(f"[INFO] Đã khôi phục model tốt nhất (dev loss = {best_dev_loss:.4f})")

        print("[INFO] Huấn luyện NER hoàn tất.")
        return self.nlp

    def _evaluate_loss(self, dev_data):
        """Tính loss trên tập dev mà KHÔNG cập nhật trọng số (để đánh giá generalization)"""
        examples = []
        for text, annotations in dev_data:
            doc = self.nlp.make_doc(text)
            examples.append(Example.from_dict(doc, annotations))
        losses = {}
        # sgd=None + drop=0.0 để chỉ đo loss, không update weights
        self.nlp.get_pipe("ner").update(examples, sgd=None, losses=losses)
        return losses.get('ner', 0.0)

    def evaluate_accuracy(self, dev_data):
        """Đánh giá độ chính xác nhận diện entity trên tập dev (đo bằng exact match)"""
        correct, total = 0, 0
        for text, annotations in dev_data:
            true_ents = set(tuple(e) for e in annotations['entities'])
            doc = self.nlp(text)
            pred_ents = set((ent.start_char, ent.end_char, ent.label_) for ent in doc.ents)
            if true_ents == pred_ents:
                correct += 1
            total += 1
        acc = correct / total if total > 0 else 0.0
        print(f"[INFO] Dev exact-match accuracy: {acc:.2%} ({correct}/{total})")
        return acc

    # ------------------------------------------------------------
    # 3. Trích xuất entity từ câu mới
    # ------------------------------------------------------------
    def extract_entities(self, text: str) -> dict:
        doc = self.nlp(text)
        result = {'RATIO': None, 'COMPANY': None}
        for ent in doc.ents:
            if ent.label_ in result:
                result[ent.label_] = ent.text
        return result

    def save(self, path="models/saved/ner_model"):
        self.nlp.to_disk(path)
        print(f"[INFO] Đã lưu NER model tại {path}")

    def load(self, path="models/saved/ner_model"):
        self.nlp = spacy.load(path)
        print(f"[INFO] Đã load NER model từ {path}")


# ============================================================
# Hàm tiện ích chạy nhanh toàn bộ pipeline train + test
# ============================================================
def run_ner_pipeline(n_train=150, n_dev=40, n_iter=30, patience=5):
    ratios_list = ['ROE', 'Return on Equity', 'PE', 'pe ratio', 'Sales',
                   'Revenue', 'CFO', 'Cash Flow']
    companies_list = ['Apple', 'Apple Inc', 'Citi', 'Citibank',
                       'Delta', 'Delta Airlines', 'BofA', 'Bank of America',
                       'Google', 'Amazon', 'Tesla', 'Microsoft']

    ner_model = FinancialNERModel()

    print(">> [NER] Đang sinh dữ liệu train/dev (tách theo tổ hợp company-ratio)...")
    train_data, dev_data = ner_model.generate_train_dev_split(
        FinancialNERModel.DEFAULT_TEMPLATES, ratios_list, companies_list,
        n_train=n_train, n_dev=n_dev
    )

    print(f">> [NER] Đang huấn luyện (tối đa {n_iter} vòng, early stopping patience={patience})...")
    ner_model.train(train_data, dev_data=dev_data, n_iter=n_iter, patience=patience)

    print("\n>> [NER] Đánh giá trên tập dev (tổ hợp company-ratio CHƯA từng thấy khi train):")
    ner_model.evaluate_accuracy(dev_data)

    test_sentences = [
        "What is the ROE for Citibank?",
        "Tell me the PE for Delta?",
        "Get me the Revenue for Apple",
        "Hello, how are you doing today?",  # câu không có entity - test negative case
    ]
    print("\n=== Kiểm tra model trên câu mới ===")
    for s in test_sentences:
        print(f"  '{s}' -> {ner_model.extract_entities(s)}")

    ner_model.save()
    return ner_model


if __name__ == "__main__":
    run_ner_pipeline()

Overwriting models/ner_model.py


In [66]:
%%writefile models/chatbot.py
"""
Chatbot kết hợp:
- ChatterBot corpus (english.greetings, english.conversations) cho tiếng Anh
- List training thủ công cho tiếng Việt
- Custom logic adapter (FinancialRatioAdapter) để trả lời câu hỏi tỷ số tài chính

Đã fix lỗi: pip package `chatterbot_corpus` thường thiếu file dữ liệu YAML
thực tế (bug đóng gói đã biết). Giải pháp: tự động clone dữ liệu gốc từ
GitHub (gunthercox/chatterbot-corpus) và trỏ trainer vào path cục bộ.
"""

import os
import subprocess
import glob

from chatterbot import ChatBot
from chatterbot.trainers import ChatterBotCorpusTrainer, ListTrainer

CORPUS_CLONE_DIR = "external_corpus/chatterbot-corpus"


def _ensure_corpus_data():
    """
    Kiểm tra xem dữ liệu corpus (file .yml) có tồn tại chưa.
    Nếu chưa -> clone trực tiếp từ GitHub về thư mục cục bộ,
    trả về đường dẫn tới folder data/english để dùng trực tiếp
    (ChatterBotCorpusTrainer chấp nhận cả dotted-path lẫn path
    file hệ thống thật).
    """
    english_dir = os.path.join(CORPUS_CLONE_DIR, "chatterbot_corpus", "data", "english")

    if os.path.exists(english_dir) and glob.glob(os.path.join(english_dir, "*.yml")):
        return english_dir

    print("[INFO] Không tìm thấy dữ liệu corpus (.yml) -> đang clone từ GitHub...")
    os.makedirs("external_corpus", exist_ok=True)
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/gunthercox/chatterbot-corpus.git",
             CORPUS_CLONE_DIR],
            check=True, capture_output=True, text=True
        )
        print("[INFO] Clone corpus thành công.")
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] Clone corpus thất bại: {e.stderr}")
        return None

    if os.path.exists(english_dir) and glob.glob(os.path.join(english_dir, "*.yml")):
        return english_dir
    print("[ERROR] Vẫn không tìm thấy dữ liệu corpus sau khi clone.")
    return None


def build_english_chatbot(name="TraderBot"):
    """Chatbot tiếng Anh dùng corpus có sẵn của ChatterBot (giống sách mục 3)"""
    chatbot = ChatBot(
        name,
        preprocessors=['chatterbot.preprocessors.clean_whitespace'],
        logic_adapters=[
            'chatterbot.logic.BestMatch',
            'chatterbot.logic.MathematicalEvaluation',
        ]
    )

    corpus_trainer = ChatterBotCorpusTrainer(chatbot)

    # ---- Thử cách chuẩn trước (dotted path qua pip package) ----
    trained_ok = False
    try:
        corpus_trainer.train("chatterbot.corpus.english")
        corpus_trainer.train("chatterbot.corpus.english.greetings")
        corpus_trainer.train("chatterbot.corpus.english.conversations")
        trained_ok = True
        print("[INFO] Train corpus thành công qua pip package chatterbot_corpus.")
    except (FileNotFoundError, OSError) as e:
        print(f"[WARN] pip package chatterbot_corpus thiếu dữ liệu ({e}). "
              f"Chuyển sang clone corpus từ GitHub...")

    # ---- Fallback: clone corpus từ GitHub, train bằng path cục bộ ----
    if not trained_ok:
        english_dir = _ensure_corpus_data()
        if english_dir:
            # Train toàn bộ folder english (bao gồm greetings, conversations, ai.yml,...)
            corpus_trainer.train(english_dir)
            print(f"[INFO] Train corpus thành công từ path cục bộ: {english_dir}")
            trained_ok = True
        else:
            print("[WARN] Không thể train corpus (thiếu dữ liệu). "
                  "Chatbot sẽ chỉ dùng list training + custom adapter.")

    list_trainer = ListTrainer(chatbot)
    list_trainer.train(['Help!', 'Please go to google.com'])
    list_trainer.train(['What is Bitcoin?', 'It is a decentralized digital currency'])

    print(f"[INFO] Đã huấn luyện chatbot tiếng Anh '{name}'.")
    return chatbot


def build_vietnamese_chatbot(name="TraderBotVN"):
    """
    Chatbot tiếng Việt - ChatterBot không có corpus tiếng Việt dựng sẵn,
    nên tự tạo tập hội thoại cơ bản bằng ListTrainer.
    """
    chatbot = ChatBot(
        name,
        preprocessors=['chatterbot.preprocessors.clean_whitespace'],
        logic_adapters=[
            'chatterbot.logic.BestMatch',
            'chatterbot.logic.MathematicalEvaluation',
        ]
    )

    list_trainer = ListTrainer(chatbot)

    basic_conversations = [
        ['Xin chào', 'Chào bạn, tôi có thể giúp gì cho bạn?'],
        ['Bạn khỏe không', 'Tôi ổn, cảm ơn bạn đã hỏi thăm'],
        ['Cảm ơn', 'Không có gì, rất vui được giúp bạn'],
        ['Tạm biệt', 'Hẹn gặp lại bạn'],
        ['Bạn là ai', 'Tôi là trợ lý ảo hỗ trợ tra cứu thông tin tài chính'],
        ['Bitcoin là gì', 'Bitcoin là một loại tiền điện tử phi tập trung'],
    ]
    for pair in basic_conversations:
        list_trainer.train(pair)

    print(f"[INFO] Đã huấn luyện chatbot tiếng Việt '{name}' với hội thoại tự tạo.")
    return chatbot


class FinancialChatbot:
    def __init__(self, lang='en', use_trained_ner=False):
        from core.financial_ratio_adapter import FinancialRatioAdapter
        self.lang = lang
        self.ratio_adapter = FinancialRatioAdapter(lang=lang, use_trained_ner=use_trained_ner)

        if lang == 'en':
            self.chatbot = build_english_chatbot()
        else:
            self.chatbot = build_vietnamese_chatbot()

    def get_response(self, user_input: str) -> str:
        ratio_response = self.ratio_adapter.process(user_input)
        fail_msgs = ["Sorry! Could not figure out what the user wants",
                     "Xin lỗi! Không xác định được yêu cầu của bạn"]
        if ratio_response not in fail_msgs:
            return ratio_response

        response = self.chatbot.get_response(user_input)
        return str(response)


if __name__ == "__main__":
    bot_en = FinancialChatbot(lang='en')
    for q in ["Hi", "What is the ROE for Apple?", "What is 78964 plus 5970"]:
        print(f"  '{q}' -> {bot_en.get_response(q)}")

Overwriting models/chatbot.py


In [67]:
%%writefile models/__init__.py
from .lstm_sentiment import LSTMSentimentModel, run_lstm_pipeline
from .lda_topic_model import LDATopicModel, run_lda_pipeline
from .ner_model import FinancialNERModel, run_ner_pipeline

Overwriting models/__init__.py


In [68]:
%%writefile core/base_pipeline.py
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np


class BaseFinancialPipeline(ABC):
    LANG = 'en'
    DEFAULT_INITIAL_CASH = 100000  # USD-scale, lớp con override cho VNĐ
    DEFAULT_LOT = 100

    def __init__(self, tickers, start, end):
        self.tickers = tickers
        self.start = start
        self.end = end
        self.stock_data = None
        self.news_data = None
        self.lexicon = None
        self._chatbot = None

    @abstractmethod
    def load_stock_data(self): pass

    @abstractmethod
    def load_news_data(self): pass

    @abstractmethod
    def preprocess_text(self, text: str) -> str: pass

    @abstractmethod
    def load_lexicon(self) -> dict: pass

    def compute_sentiment_score(self, text: str) -> float:
        if self.lexicon is None:
            self.lexicon = self.load_lexicon()
        tokens = self.preprocess_text(text).split()
        return sum(self.lexicon.get(t, 0.0) for t in tokens)

    def run_case_study_1(self, signal_mode="sign",
                          quantile_pct=0.25, fixed_threshold=0.5,
                          liquidate_at_end=True,
                          initial_cash=None, lot=None):
        print(">> [Case Study 1] Sentiment Analysis Trading Strategy")
        if self.stock_data is None:
            self.load_stock_data()
        if self.news_data is None:
            self.load_news_data()

        if self.news_data.empty or 'Close' not in self.news_data.columns:
            print("[WARN] Thiếu dữ liệu giá (Close) để backtest.")
            return self.news_data

        # FIX: Dùng vốn/lot mặc định theo từng pipeline (VNĐ cần vốn lớn hơn
        # USD rất nhiều do thang giá khác nhau - 1 lot = 100 cổ phiếu VN giá
        # ~10,000-130,000 VNĐ cần 1-13 triệu VNĐ/lệnh, trong khi 100,000
        # "tiền" chỉ đủ cho thang giá USD của cổ phiếu Mỹ).
        initial_cash = initial_cash if initial_cash is not None else self.DEFAULT_INITIAL_CASH
        lot = lot if lot is not None else self.DEFAULT_LOT
        print(f"[INFO] Vốn ban đầu mỗi mã: {initial_cash:,.0f}, lot size: {lot}")

        df = self.news_data.copy()
        df['sentiment'] = df['headline'].apply(self.compute_sentiment_score)

        n_zero = (df['sentiment'] == 0).sum()
        n_pos = (df['sentiment'] > 0).sum()
        n_neg = (df['sentiment'] < 0).sum()
        print(f"[DEBUG] Phân bố sentiment: min={df['sentiment'].min():.3f}, "
              f"max={df['sentiment'].max():.3f}, mean={df['sentiment'].mean():.3f}")
        print(f"[DEBUG] Số dòng: dương={n_pos}, âm={n_neg}, bằng 0={n_zero}/{len(df)}")

        if signal_mode == "sign":
            df['signal'] = df['sentiment'].apply(
                lambda s: 'BUY' if s > 0 else ('SELL' if s < 0 else 'HOLD')
            )
        elif signal_mode == "quantile":
            nonzero = df[df['sentiment'] != 0]['sentiment']
            if len(nonzero) < 2 or nonzero.nunique() < 2:
                df['signal'] = df['sentiment'].apply(
                    lambda s: 'BUY' if s > 0 else ('SELL' if s < 0 else 'HOLD')
                )
            else:
                upper = nonzero.quantile(1 - quantile_pct)
                lower = nonzero.quantile(quantile_pct)
                df['signal'] = df['sentiment'].apply(
                    lambda s: 'BUY' if s >= upper else ('SELL' if s <= lower else 'HOLD')
                )
        else:
            df['signal'] = df['sentiment'].apply(
                lambda s: 'BUY' if s > fixed_threshold
                else ('SELL' if s < -fixed_threshold else 'HOLD')
            )

        n_buy = (df['signal'] == 'BUY').sum()
        n_sell = (df['signal'] == 'SELL').sum()
        print(f"[INFO] Số tín hiệu: BUY={n_buy}, SELL={n_sell}, "
              f"HOLD={len(df) - n_buy - n_sell}")

        if 'ticker' in df.columns:
            print("\n[DIAGNOSTIC] Chi tiết theo từng ticker (kèm giá min-max để "
                  "xác nhận biến động giá):")
            print(f"{'Ticker':<8}{'Tổng dòng':<11}{'Có giá':<9}"
                  f"{'BUY':<6}{'SELL':<6}{'Giá min':<12}{'Giá max':<12}"
                  f"{'1 lot cần':<14}")
            for ticker in sorted(df['ticker'].unique()):
                sub = df[df['ticker'] == ticker]
                n_total = len(sub)
                n_has_price = sub['Close'].notna().sum()
                n_buy_t = (sub['signal'] == 'BUY').sum()
                n_sell_t = (sub['signal'] == 'SELL').sum()
                prices = sub['Close'].dropna()
                pmin = prices.min() if len(prices) else 0
                pmax = prices.max() if len(prices) else 0
                cost_per_lot = pmin * lot
                afford_flag = "✓" if initial_cash >= cost_per_lot else "✗ KHÔNG ĐỦ VỐN"
                print(f"{ticker:<8}{n_total:<11}{n_has_price:<9}"
                      f"{n_buy_t:<6}{n_sell_t:<6}{pmin:<12,.0f}{pmax:<12,.0f}"
                      f"{cost_per_lot:<14,.0f}{afford_flag}")

        if 'ticker' in df.columns:
            results_per_ticker = {}
            trade_summary = {}
            all_results = []
            all_trade_logs = {}
            for ticker in df['ticker'].unique():
                sub = df[df['ticker'] == ticker].copy()
                if sub.empty:
                    continue
                sub_result, summary, trade_log = self._backtest_simple(
                    sub, initial_cash=initial_cash, lot=lot,
                    liquidate_at_end=liquidate_at_end
                )
                all_results.append(sub_result)
                final_value = sub_result['portfolio_value'].iloc[-1]
                results_per_ticker[ticker] = final_value
                trade_summary[ticker] = summary
                all_trade_logs[ticker] = trade_log

            result = pd.concat(all_results, ignore_index=True) if all_results else df

            # ---- In log giao dịch chi tiết theo yêu cầu: xem giá tại mỗi lần mua/bán ----
            print("\n[TRADE LOG] Chi tiết từng lệnh đã thực thi (để xác nhận biến động giá):")
            any_trade_executed = False
            for ticker, log in all_trade_logs.items():
                if log:
                    any_trade_executed = True
                    print(f"\n  {ticker}:")
                    for entry in log:
                        print(f"    {entry['action']:<5} @ {entry['price']:>12,.0f} VNĐ/USD "
                              f"(dòng {entry['row_idx']})")
            if not any_trade_executed:
                print("  (Không có lệnh nào được thực thi - kiểm tra bảng DIAGNOSTIC "
                      "phía trên cột 'Giá min' vs '1 lot cần' vs vốn ban đầu)")

            print("\n[KẾT QUẢ LÃI/LỖ] (đã thanh lý toàn bộ vị thế cuối kỳ):")
            print(f"{'Ticker':<8}{'Vốn đầu':<16}{'Kết thúc':<16}"
                  f"{'Lãi/Lỗ':<16}{'% Lãi/Lỗ':<10}{'Số lệnh':<10}")
            for ticker, final_value in results_per_ticker.items():
                summary = trade_summary.get(ticker, {})
                ic = summary.get('initial_cash', initial_cash)
                pnl = final_value - ic
                pnl_pct = (pnl / ic) * 100 if ic else 0
                n_trades = len(all_trade_logs.get(ticker, []))
                status = "LÃI" if pnl > 0 else ("LỖ" if pnl < 0 else "HÒA VỐN")
                print(f"{ticker:<8}{ic:<16,.0f}{final_value:<16,.0f}"
                      f"{pnl:<+16,.0f}{pnl_pct:<+9.2f}%{n_trades:<10}[{status}]")

            self.news_data = result
            return result, results_per_ticker
        else:
            result, _, _ = self._backtest_simple(
                df, initial_cash=initial_cash, lot=lot, liquidate_at_end=liquidate_at_end
            )
            self.news_data = result
            return result

    def _backtest_simple(self, df, initial_cash=100000, lot=100, liquidate_at_end=True):
        """
        Backtest đơn giản với BUY/SELL theo tín hiệu sentiment.
        Trả về (df_với_portfolio_value, summary_dict, trade_log_list)
        trade_log_list: list các dict {action, price, row_idx} cho MỌI lệnh
        THỰC SỰ được thực thi (không phải mọi tín hiệu, vì có thể bị chặn
        do thiếu vốn/thiếu vị thế).
        """
        cash, position, portfolio_values = initial_cash, 0, []
        last_valid_price = None
        trade_log = []

        for idx, row in df.iterrows():
            price = row.get('Close', np.nan)
            if pd.isna(price):
                portfolio_values.append(cash + position * (last_valid_price or 0))
                continue

            last_valid_price = price
            if row['signal'] == 'BUY' and cash >= price * lot:
                cash -= price * lot
                position += lot
                trade_log.append({'action': 'BUY', 'price': price, 'row_idx': idx})
            elif row['signal'] == 'SELL' and position >= lot:
                cash += price * lot
                position -= lot
                trade_log.append({'action': 'SELL', 'price': price, 'row_idx': idx})
            portfolio_values.append(cash + position * price)

        remaining_position_before_liquidation = position

        if liquidate_at_end and position > 0 and last_valid_price is not None:
            cash += position * last_valid_price
            trade_log.append({'action': 'LIQUIDATE', 'price': last_valid_price, 'row_idx': 'end'})
            position = 0
            if portfolio_values:
                portfolio_values[-1] = cash

        df = df.copy()
        df['portfolio_value'] = portfolio_values

        summary = {
            'initial_cash': initial_cash,
            'final_cash': cash,
            'remaining_position_before_liquidation': remaining_position_before_liquidation,
            'liquidated': liquidate_at_end and remaining_position_before_liquidation > 0,
        }
        return df, summary, trade_log

    def run_case_study_2(self, user_input: str = None, use_trained_ner=False):
        print(">> [Case Study 2] Chatbot Financial Ratio Assistant")
        from models.chatbot import FinancialChatbot
        if self._chatbot is None:
            self._chatbot = FinancialChatbot(lang=self.LANG, use_trained_ner=use_trained_ner)
        if user_input is None:
            user_input = ("What is the ROE for Apple?" if self.LANG == 'en'
                          else "Cho tôi xem tỷ số ROE của Vinamilk")
        result = self._chatbot.get_response(user_input)
        print(f"  Input : {user_input}")
        print(f"  Output: {result}")
        return result

    def run_case_study_3(self, documents: list, n_topics=5, n_top_words=10, visualize=False):
        print(">> [Case Study 3] Document Topic Modeling (LDA)")
        from models.lda_topic_model import run_lda_pipeline
        stop_words = 'english' if self.LANG == 'en' else None
        model, topics, panel = run_lda_pipeline(
            documents, n_topics=n_topics, n_words=n_top_words,
            visualize=visualize, stop_words=stop_words
        )
        return model, topics, panel

Overwriting core/base_pipeline.py


In [69]:
%%writefile core/english_pipeline.py
"""
Pipeline tiếng Anh - bám theo Case Study 1 (Chapter 10) của sách
"Machine Learning and Data Science Blueprints for Finance".
"""

import os
import json
import zipfile
import warnings
import numpy as np
import pandas as pd

import yfinance as yf
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

from core.base_pipeline import BaseFinancialPipeline

warnings.filterwarnings("ignore")
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)


class EnglishBookPipeline(BaseFinancialPipeline):

    LANG = 'en'

    def __init__(self,
                 tickers=None,
                 start='2010-01-01',
                 end='2018-12-31',
                 headline_zip_path="data/raw/Raw Headline Data.zip",
                 stock_lex_path="data/lexicons/stock_lex.csv"):
        default_tickers = ['AAPL', 'MSFT', 'AMZN', 'GOOG', 'FB',
                            'WMT', 'JPM', 'TSLA', 'NFLX', 'ADBE']
        super().__init__(tickers or default_tickers, start, end)

        self.headline_zip_path = headline_zip_path
        self.stock_lex_path = stock_lex_path

        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
            self.nlp = spacy.load("en_core_web_sm")

        self.stop_words = set(stopwords.words('english'))
        self.sia = SentimentIntensityAnalyzer()
        self.lstm_model = None

    def load_stock_data(self):
        print(">> [2.2] Đang tải dữ liệu giá cổ phiếu từ Yahoo Finance...")
        df_ticker_return = pd.DataFrame()
        for ticker in self.tickers:
            try:
                ticker_yf = yf.Ticker(ticker)
                data_temp = ticker_yf.history(start=self.start, end=self.end)
                data_temp['ticker'] = ticker
                data_temp = data_temp.reset_index()
                df_ticker_return = pd.concat([df_ticker_return, data_temp], ignore_index=True)
            except Exception as e:
                print(f"[WARN] Lỗi tải {ticker}: {e}")

        self.stock_data = df_ticker_return
        os.makedirs("data/processed", exist_ok=True)
        self.stock_data.to_csv("data/processed/Step3.2_ReturnData.csv", index=False)
        print(f"[INFO] Đã tải xong {len(self.stock_data)} dòng giá cổ phiếu.")
        return self.stock_data

    def _json_parser(self, json_data):
        from lxml import etree
        from io import StringIO
        xml_data = json_data['content']
        try:
            tree = etree.parse(StringIO(xml_data), parser=etree.HTMLParser())
        except Exception:
            return [], []

        headlines = tree.xpath("//h4[contains(@class, 'media-heading')]/a/text()")
        main_tickers = list(map(lambda x: x.replace('/symbol/', ''),
                                 tree.xpath("//div[contains(@class, 'media-left')]//a/@href")))
        final_headlines = [''.join(f.xpath('.//text()')) for f in
                            tree.xpath("//div[contains(@class, 'media-body')]/ul/li[1]")]
        if len(final_headlines) == 0:
            final_headlines = [''.join(f.xpath('.//text()')) for f in
                                tree.xpath("//div[contains(@class, 'media-body')]")]
        final_headlines = [f.replace(h, '').split('\xa0')[0].strip()
                            for f, h in zip(final_headlines, headlines)]
        return main_tickers, final_headlines

    def load_news_data(self):
        print(">> [3.1] Đang xử lý dữ liệu tin tức (JSON headlines)...")
        rows = []
        try:
            z = zipfile.ZipFile(self.headline_zip_path, "r")
            for fname in z.namelist():
                try:
                    file_data = z.open(fname).read()
                    json_data = json.loads(file_data)
                    date_str = fname.split('/')[-1].replace('.json', '')
                    tickers_parsed, headlines_parsed = self._json_parser(json_data)
                    for t, h in zip(tickers_parsed, headlines_parsed):
                        rows.append({'ticker': t, 'headline': h, 'date': date_str})
                except Exception:
                    continue
            data_df_news = pd.DataFrame(rows)
        except FileNotFoundError:
            print(f"[WARN] Không tìm thấy {self.headline_zip_path}. Dùng dữ liệu mẫu demo.")
            data_df_news = pd.DataFrame({
                'ticker': ['AAPL', 'GOOG', 'AMZN', 'TSLA'],
                'headline': [
                    "Apple (AAPL) jumps after record iPhone sales beat estimates",
                    "Google (GOOG) plunge on antitrust investigation concerns",
                    "Amazon (AMZN) posts strong earnings, shares surge",
                    "Tesla (TSLA) falls after production delay announcement"
                ],
                'date': pd.date_range('2018-01-01', periods=4).astype(str)
            })

        if self.stock_data is None:
            self.load_stock_data()

        stock_df = self.stock_data.copy()
        stock_df['date'] = pd.to_datetime(stock_df['Date']).dt.strftime('%Y-%m-%d')
        stock_df = stock_df.sort_values(['ticker', 'date'])
        stock_df['ret_curr'] = stock_df.groupby('ticker')['Close'].pct_change()
        stock_df['eventRet'] = (stock_df['ret_curr']
                                 + stock_df.groupby('ticker')['ret_curr'].shift(-1)
                                 + stock_df.groupby('ticker')['ret_curr'].shift(1))

        data_df_news['date'] = pd.to_datetime(data_df_news['date'], errors='coerce').dt.strftime('%Y-%m-%d')
        combined = pd.merge(data_df_news, stock_df, how='left', on=['date', 'ticker'])
        combined = combined[combined['ticker'].isin(self.tickers)]
        combined = combined[['ticker', 'headline', 'date', 'eventRet', 'Close']].dropna(subset=['headline'])

        self.news_data = combined
        print(f"[INFO] Số dòng dữ liệu tin tức đã ghép với giá: {len(self.news_data)}")
        return self.news_data

    def preprocess_text(self, text: str) -> str:
        doc = self.nlp(str(text).lower())
        tokens = [t.text for t in doc if t.text not in self.stop_words and t.is_alpha]
        return " ".join(tokens)

    def load_lexicon(self) -> dict:
        print(">> [4.3] Đang xây dựng lexicon (VADER + stock_lex.csv)...")
        try:
            stock_lex = pd.read_csv(self.stock_lex_path)
            stock_lex['sentiment'] = (stock_lex['Aff_Score'] + stock_lex['Neg_Score']) / 2
            stock_lex = dict(zip(stock_lex['Item'], stock_lex['sentiment']))
            stock_lex = {k: v for k, v in stock_lex.items() if len(str(k).split(' ')) == 1}

            max_v, min_v = max(stock_lex.values()), min(stock_lex.values())
            stock_lex_scaled = {
                k: (v / max_v * 4 if v > 0 else v / min_v * -4)
                for k, v in stock_lex.items()
            }

            final_lex = {}
            final_lex.update(stock_lex_scaled)
            final_lex.update(self.sia.lexicon)
            self.sia.lexicon = final_lex
            self.lexicon = final_lex
            print(f"[INFO] Lexicon tổng hợp có {len(final_lex)} từ.")
        except FileNotFoundError:
            print(f"[WARN] Không tìm thấy {self.stock_lex_path}. Dùng VADER lexicon mặc định.")
            self.lexicon = self.sia.lexicon
        return self.lexicon

    def compute_sentiment_score(self, text: str) -> float:
        if self.lexicon is None:
            self.load_lexicon()
        return self.sia.polarity_scores(str(text))['compound']

    def compute_sentiment_textblob(self, text: str) -> float:
        return TextBlob(str(text)).sentiment.polarity

    def compute_sentiment_lstm(self, texts: list):
        """
        FIX: trả về XÁC SUẤT liên tục (probs) thay vì nhãn nhị phân (0/1).
        Lý do: nếu dùng nhãn 0/1 và model dự đoán toàn bộ 1 lớp duy nhất
        (rất dễ xảy ra khi model train trên tập dữ liệu có phân bố khác
        hẳn với dữ liệu áp dụng), cột đó có phương sai = 0, khiến
        pandas.corr() trả về NaN cho toàn bộ hàng/cột liên quan - đây là
        quy tắc toán học chuẩn (Pearson correlation chia cho std=0),
        không phải bug. Dùng xác suất liên tục giúp giữ được biến động
        nhỏ, tránh tình trạng NaN toàn phần.
        """
        from models.lstm_sentiment import LSTMSentimentModel
        if self.lstm_model is None:
            self.lstm_model = LSTMSentimentModel()
            try:
                self.lstm_model.load("models/saved/lstm_sentiment.h5")
            except Exception as e:
                print(f"[WARN] Chưa có LSTM model đã train ({e}).")
                return np.full(len(texts), np.nan)
        preds, probs = self.lstm_model.predict(texts)
        return probs  # dùng xác suất (0.0 - 1.0) thay vì nhãn 0/1

    def run_case_study_1(self, use_lstm=False, liquidate_at_end=True):
        print(">> [Case Study 1] NLP & Sentiment Analysis-Based Trading Strategy")
        if self.stock_data is None:
            self.load_stock_data()
        if self.news_data is None:
            self.load_news_data()
        if self.lexicon is None:
            self.load_lexicon()

        df = self.news_data.copy()
        df['sentiment_textblob'] = df['headline'].apply(self.compute_sentiment_textblob)
        df['sentiment_lex'] = df['headline'].apply(self.compute_sentiment_score)

        if use_lstm:
            lstm_scores = self.compute_sentiment_lstm(df['headline'].tolist())
            df['sentiment_lstm'] = lstm_scores

            # Kiểm tra phương sai trước khi tính correlation để tránh NaN gây khó hiểu
            lstm_std = df['sentiment_lstm'].std()
            if pd.isna(lstm_std) or lstm_std == 0:
                print(f"[WARN] LSTM sentiment có phương sai = 0 (model dự đoán gần như "
                      f"cùng 1 giá trị cho mọi headline). Tương quan sẽ là NaN - đây là "
                      f"hạn chế của model, không phải lỗi tính toán. Std hiện tại: {lstm_std}")
            else:
                print(f"[INFO] LSTM sentiment - std: {lstm_std:.4f}, "
                      f"min: {df['sentiment_lstm'].min():.4f}, "
                      f"max: {df['sentiment_lstm'].max():.4f}")

        print("\n[INFO] Tương quan sentiment vs eventRet:")
        cols = ['sentiment_textblob', 'sentiment_lex', 'eventRet']
        if use_lstm:
            cols.insert(2, 'sentiment_lstm')
        print(df[cols].corr())

        df = df.sort_values(['ticker', 'date'])
        df['prev_sentiment'] = df.groupby('ticker')['sentiment_lex'].shift(1)
        df['sentiment_change'] = df['sentiment_lex'] - df['prev_sentiment']

        df['signal'] = 'HOLD'
        df.loc[df['sentiment_change'] >= 0.5, 'signal'] = 'BUY'
        df.loc[df['sentiment_change'] <= -0.5, 'signal'] = 'SELL'

        results = {}
        trade_summary = {}
        all_results = []
        for ticker in self.tickers:
            sub = df[df['ticker'] == ticker].copy()
            if sub.empty:
                continue
            # FIX: _backtest_simple giờ trả về 3 giá trị (df, summary, trade_log)
            sub_result, summary, trade_log = self._backtest_simple(sub, liquidate_at_end=liquidate_at_end)
            all_results.append(sub_result)
            final_value = sub_result['portfolio_value'].iloc[-1] if len(sub_result) else np.nan
            results[ticker] = final_value
            trade_summary[ticker] = summary
            print(f"  {ticker}: Ending Portfolio Value = {final_value:.2f}")

        print("\n[KẾT QUẢ LÃI/LỖ] (đã thanh lý toàn bộ vị thế cuối kỳ):")
        print(f"{'Ticker':<8}{'Vốn đầu':<14}{'Kết thúc':<14}"
              f"{'Lãi/Lỗ':<14}{'% Lãi/Lỗ':<10}")
        for ticker, final_value in results.items():
            if pd.isna(final_value):
                continue
            summary = trade_summary.get(ticker, {})
            initial_cash = summary.get('initial_cash', 100000)
            pnl = final_value - initial_cash
            pnl_pct = (pnl / initial_cash) * 100
            status = "LÃI" if pnl > 0 else ("LỖ" if pnl < 0 else "HÒA VỐN")
            print(f"{ticker:<8}{initial_cash:<14,.0f}{final_value:<14,.0f}"
                  f"{pnl:<+14,.0f}{pnl_pct:<+9.2f}% [{status}]")

        result = pd.concat(all_results, ignore_index=True) if all_results else df
        self.news_data = result
        return result, results


if __name__ == "__main__":
    pipeline = EnglishBookPipeline()
    pipeline.run_case_study_1()

Overwriting core/english_pipeline.py


In [70]:
%%writefile core/vietnamese_pipeline.py
import os
import re
import numpy as np
import pandas as pd
from underthesea import word_tokenize

from core.base_pipeline import BaseFinancialPipeline

COMPANY_TO_TICKER = {
    'vinamilk': 'VNM', 'sữa việt nam': 'VNM',
    'fpt': 'FPT',
    'vietcombank': 'VCB', 'vcb': 'VCB', 'ngân hàng ngoại thương': 'VCB',
    'vietinbank': 'CTG', 'ctg': 'CTG', 'ngân hàng công thương': 'CTG',
    'bidv': 'BID', 'đầu tư và phát triển': 'BID',
    'ngân hàng đầu tư và phát triển': 'BID',
    'hòa phát': 'HPG', 'hoa phat': 'HPG', 'tập đoàn hòa phát': 'HPG',
    'acb': 'ACB', 'ngân hàng á châu': 'ACB',
    'mb bank': 'MBB', 'mbbank': 'MBB', 'quân đội': 'MBB',
    'ngân hàng quân đội': 'MBB',
    'nam kim': 'NKG', 'thép nam kim': 'NKG',
    'ree': 'REE', 'cơ điện lạnh': 'REE',
    'pv gas': 'GAS', 'khí việt nam': 'GAS', 'tổng công ty khí việt nam': 'GAS',
    'vingroup': 'VIC', 'vin group': 'VIC',
}

CSV_TICKERS = {'ACB', 'BID', 'CTG', 'FPT', 'GAS', 'HPG', 'MBB', 'NKG', 'REE', 'VCB'}
SUPPLEMENTARY_TICKERS = {'VNM', 'VIC'}


class VietnameseRealDataPipeline(BaseFinancialPipeline):

    LANG = 'vi'
    DEFAULT_INITIAL_CASH = 100_000_000  # 100 triệu VNĐ - đủ mua nhiều lot ở thang giá VN
    DEFAULT_LOT = 100

    def __init__(self, tickers, start, end,
                 news_csv_path="data/processed/processed_news.csv",
                 stopwords_path="data/lexicons/vietnamese-stopwords.txt",
                 lexicon_path="data/lexicons/viet_lexicon.csv",
                 price_csv_path="data/raw/data_10y.csv",
                 manual_csv_dir="data/raw/vn_prices",
                 try_supplementary=True):
        super().__init__(tickers, start, end)
        self.news_csv_path = news_csv_path
        self.stopwords_path = stopwords_path
        self.lexicon_path = lexicon_path
        self.price_csv_path = price_csv_path
        self.manual_csv_dir = manual_csv_dir
        self.try_supplementary = try_supplementary
        self.stop_words = self._load_stopwords()
        self._available_tickers = None

    def _load_stopwords(self):
        try:
            with open(self.stopwords_path, encoding='utf-8') as f:
                raw_words = [line.strip() for line in f if line.strip()]
            stopwords_set = set(w.replace(" ", "_") for w in raw_words)
            stopwords_set.update(raw_words)
            print(f"[INFO] Đã load {len(raw_words)} stopwords từ {self.stopwords_path}")
            return stopwords_set
        except Exception as e:
            print(f"[WARN] Không load được stopwords: {e}")
            return set()

    def _load_from_multiindex_csv(self, path):
        raw = pd.read_csv(path, header=[0, 1], index_col=0)
        raw.index.name = 'Date'
        raw = raw[raw.index.astype(str) != 'Date']
        raw.index = pd.to_datetime(raw.index, errors='coerce')
        raw = raw[raw.index.notna()]

        if raw.empty:
            return pd.DataFrame()

        tickers_in_csv = sorted(set(col[1] for col in raw.columns if col[1] != ''))

        frames = []
        for full_ticker in tickers_in_csv:
            try:
                sub = pd.DataFrame({
                    'Date': raw.index,
                    'Open': pd.to_numeric(raw[('Open', full_ticker)], errors='coerce').values,
                    'High': pd.to_numeric(raw[('High', full_ticker)], errors='coerce').values,
                    'Low': pd.to_numeric(raw[('Low', full_ticker)], errors='coerce').values,
                    'Close': pd.to_numeric(raw[('Close', full_ticker)], errors='coerce').values,
                    'Volume': pd.to_numeric(raw[('Volume', full_ticker)], errors='coerce').values,
                })
                clean_ticker = full_ticker.replace('.VN', '').replace('.HM', '').replace('.HN', '')
                sub['ticker'] = clean_ticker
                sub = sub.dropna(subset=['Close'])
                frames.append(sub)
            except KeyError as e:
                print(f"[WARN] Thiếu cột dữ liệu cho ticker {full_ticker}: {e}")
                continue

        if not frames:
            return pd.DataFrame()

        result = pd.concat(frames, ignore_index=True)
        result = result[
            (result['Date'] >= pd.to_datetime(self.start)) &
            (result['Date'] <= pd.to_datetime(self.end))
        ]
        return result

    def _fetch_supplementary_prices(self, missing_tickers):
        if not missing_tickers:
            return pd.DataFrame()

        import yfinance as yf
        print(f"[INFO] Đang thử bổ sung giá cho các mã: {sorted(missing_tickers)} "
              f"(được nhắc nhiều trong tin tức nhưng không có trong CSV có sẵn)...")

        suffixes_to_try = ['.VN', '.HM', '.HN', '']
        frames = []
        for t in missing_tickers:
            fetched = False
            for suffix in suffixes_to_try:
                yf_ticker = f"{t}{suffix}"
                try:
                    df = yf.download(yf_ticker, start=self.start, end=self.end, progress=False)
                    if df is not None and not df.empty:
                        df = df.reset_index()
                        df['ticker'] = t
                        frames.append(df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ticker']])
                        print(f"  ✓ Tải thành công {yf_ticker}: {len(df)} dòng")
                        fetched = True
                        break
                except Exception:
                    continue
            if not fetched:
                print(f"  ✗ Không tải được giá cho {t}. Sẽ loại {t} khỏi ánh xạ ticker.")

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def load_stock_data(self):
        base_data = pd.DataFrame()

        if os.path.exists(self.price_csv_path):
            print(f"[INFO] Đang đọc file giá có sẵn: {self.price_csv_path}")
            try:
                base_data = self._load_from_multiindex_csv(self.price_csv_path)
                if not base_data.empty:
                    print(f"[INFO] Đã đọc {len(base_data)} dòng, "
                          f"{base_data['ticker'].nunique()} mã: {sorted(base_data['ticker'].unique())}")
            except Exception as e:
                print(f"[WARN] Lỗi đọc {self.price_csv_path}: {e}")

        if self.try_supplementary:
            existing_tickers = set(base_data['ticker'].unique()) if not base_data.empty else set()
            missing = SUPPLEMENTARY_TICKERS - existing_tickers
            supp_data = self._fetch_supplementary_prices(missing)
            if not supp_data.empty:
                base_data = pd.concat([base_data, supp_data], ignore_index=True)

        if base_data.empty:
            print("[WARN] KHÔNG tải được giá cổ phiếu VN từ bất kỳ nguồn nào.")
            base_data = pd.DataFrame(columns=['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ticker'])

        self.stock_data = base_data
        self._available_tickers = set(base_data['ticker'].unique()) if not base_data.empty else set()
        return self.stock_data

    def _detect_ticker(self, title: str):
        title_lower = str(title).lower()
        for name, ticker in COMPANY_TO_TICKER.items():
            if name in title_lower:
                if self._available_tickers is None or ticker in self._available_tickers:
                    return ticker
                else:
                    return None
        return None

    def load_news_data(self):
        try:
            df = pd.read_csv(self.news_csv_path, encoding="utf-8-sig")
            df = df.rename(columns={"title": "headline"})
        except Exception as e:
            print(f"[WARN] load_news_data (VN) lỗi: {e}")
            df = pd.DataFrame({'headline': ["Vinamilk báo lãi tăng trưởng mạnh trong quý này"]})

        if self.stock_data is None:
            self.load_stock_data()

        n_total_headlines = len(df)
        df['ticker'] = df['headline'].apply(self._detect_ticker)

        df_all_mentions = df.copy()
        df_all_mentions['ticker_mentioned'] = df_all_mentions['headline'].apply(
            lambda t: next((tk for name, tk in COMPANY_TO_TICKER.items()
                            if name in str(t).lower()), None)
        )
        n_mentioned_no_price = ((df_all_mentions['ticker_mentioned'].notna()) &
                                 (df['ticker'].isna())).sum()
        if n_mentioned_no_price > 0:
            excluded_tickers = sorted(set(
                df_all_mentions.loc[
                    (df_all_mentions['ticker_mentioned'].notna()) & (df['ticker'].isna()),
                    'ticker_mentioned'
                ]
            ))
            print(f"[INFO] Đã loại {n_mentioned_no_price} bài báo nhắc tới các mã "
                  f"KHÔNG có giá khả dụng: {excluded_tickers}")

        df = df.dropna(subset=['ticker'])
        print(f"[INFO] Số bài báo gán được ticker (có giá hợp lệ): {len(df)}/{n_total_headlines}")

        if df.empty:
            self.news_data = df
            return self.news_data

        # ----------------------------------------------------------------
        # FIX QUAN TRỌNG: gán giá theo VỊ TRÍ THỜI GIAN (không phải trung
        # bình cố định) - mỗi bài báo của cùng 1 ticker sẽ lấy giá tại các
        # thời điểm khác nhau trải dài trong toàn bộ lịch sử giá của mã đó.
        # Điều này tạo ra biến động giá thực -> mua/bán ở các mức giá khác
        # nhau -> có thể LÃI hoặc LỖ thực sự, thay vì luôn hòa vốn tuyệt
        # đối như khi dùng giá trung bình cố định cho mọi giao dịch.
        # ----------------------------------------------------------------
        df['Close'] = np.nan
        if self.stock_data is not None and not self.stock_data.empty:
            for ticker in df['ticker'].unique():
                ticker_prices = self.stock_data[
                    self.stock_data['ticker'] == ticker
                ].sort_values('Date')['Close'].values

                idx = df[df['ticker'] == ticker].index
                n = len(idx)
                if len(ticker_prices) > 0 and n > 0:
                    sampled_positions = np.linspace(0, len(ticker_prices) - 1, n).astype(int)
                    df.loc[idx, 'Close'] = ticker_prices[sampled_positions]

        self.news_data = df
        n_valid = df['Close'].notna().sum()
        print(f"[INFO] Số dòng có giá Close hợp lệ: {n_valid}/{len(df)}")
        if n_valid > 0:
            print(f"[DEBUG] Khoảng giá theo ticker (min-max, để xác nhận giá đã biến động):")
            for ticker in df['ticker'].unique():
                prices = df[df['ticker'] == ticker]['Close'].dropna()
                if len(prices) > 0:
                    print(f"  {ticker}: min={prices.min():.2f}, max={prices.max():.2f}, "
                          f"số dòng={len(prices)}")
        return self.news_data

    def preprocess_text(self, text: str) -> str:
        tokens = word_tokenize(str(text).lower(), format="text").split()
        tokens = [t for t in tokens if t not in self.stop_words]
        return " ".join(tokens)

    def load_lexicon(self) -> dict:
        try:
            df = pd.read_csv(self.lexicon_path)
            self.lexicon = dict(zip(df.iloc[:, 0], df.iloc[:, 1]))
            print(f"[INFO] Đã load lexicon VN với {len(self.lexicon)} từ.")
        except Exception as e:
            print(f"[WARN] load_lexicon (VN) lỗi: {e}")
            self.lexicon = {
                "tăng_trưởng": 1.0, "lợi_nhuận": 0.8,
                "thua_lỗ": -1.0, "phá_sản": -1.5
            }
        return self.lexicon

    def run_case_study_3(self, documents: list, n_topics=5, n_top_words=10, visualize=False):
        print(">> [Case Study 3 - VN] Document Topic Modeling (LDA) tiếng Việt")
        processed_docs = [self.preprocess_text(doc) for doc in documents]
        processed_docs = [d for d in processed_docs if d.strip() != ""]
        return super().run_case_study_3(
            processed_docs, n_topics=n_topics, n_top_words=n_top_words,
            visualize=visualize
        )

Overwriting core/vietnamese_pipeline.py


In [71]:
%%writefile core/financial_ratio_adapter.py
"""
Custom logic adapter cho Case Study 2 - hỗ trợ cả tiếng Anh và tiếng Việt.
"""

# ============================================================
# Dữ liệu ánh xạ - Tiếng Anh
# ============================================================
companies_en = {
    'AAPL': ['Apple', 'Apple Inc'],
    'GOOG': ['Google'],
    'C': ['Citi', 'Citibank'],
    'DAL': ['Delta', 'Delta Airlines'],
    'BAC': ['BAML', 'BofA', 'Bank of America'],
}

ratios_en = {
    'return-on-equity-ttm': ['ROE', 'Return on Equity'],
    'pe-ratio-ttm': ['PE', 'pe ratio', 'Price to equity'],
    'revenue-ttm': ['Sales', 'Revenue'],
    'cash-from-operations-quarterly': ['CFO', 'Cash Flow from Operations'],
}

# ============================================================
# Dữ liệu ánh xạ - Tiếng Việt
# ============================================================
companies_vi = {
    'VNM': ['Vinamilk', 'Sữa Việt Nam'],
    'FPT': ['FPT'],
    'VCB': ['Vietcombank', 'VCB'],
    'HPG': ['Hòa Phát', 'Hoa Phat'],
    'VIC': ['Vingroup', 'VinGroup'],
}

ratios_vi = {
    'return-on-equity-ttm': ['ROE', 'Tỷ suất lợi nhuận trên vốn chủ sở hữu'],
    'pe-ratio-ttm': ['PE', 'Tỷ số giá trên thu nhập', 'Hệ số P/E'],
    'revenue-ttm': ['Doanh thu', 'Doanh số'],
    'cash-from-operations-quarterly': ['Dòng tiền hoạt động', 'CFO'],
}


class FinancialRatioAdapter:
    """
    Adapter dùng chung cho cả 2 ngôn ngữ.
    lang='en' -> dùng companies_en/ratios_en
    lang='vi' -> dùng companies_vi/ratios_vi
    """

    def __init__(self, lang='en', use_trained_ner=False,
                 ner_model_path="models/saved/ner_model"):
        self.lang = lang
        if lang == 'vi':
            companies, ratios = companies_vi, ratios_vi
        else:
            companies, ratios = companies_en, ratios_en

        self.companies_rev = {v: k for k, vs in companies.items() for v in vs}
        self.ratios_rev = {v: k for k, vs in ratios.items() for v in vs}

        self.use_trained_ner = use_trained_ner
        self.ner_model = None
        if use_trained_ner:
            try:
                from models.ner_model import FinancialNERModel
                self.ner_model = FinancialNERModel()
                self.ner_model.load(ner_model_path)
            except Exception as e:
                print(f"[WARN] Không load được NER model đã train ({e}). "
                      f"Chuyển sang keyword-matching.")
                self.use_trained_ner = False

    def process(self, user_input: str) -> str:
        found_company, found_ratio = None, None

        if self.use_trained_ner and self.ner_model is not None:
            entities = self.ner_model.extract_entities(user_input)
            found_ratio = entities.get('RATIO')
            found_company = entities.get('COMPANY')
        else:
            text_lower = user_input.lower()
            for name in self.companies_rev:
                if name.lower() in text_lower:
                    found_company = name
                    break
            for name in self.ratios_rev:
                if name.lower() in text_lower:
                    found_ratio = name
                    break

        if found_company in self.companies_rev and found_ratio in self.ratios_rev:
            ticker = self.companies_rev[found_company]
            ratio_code = self.ratios_rev[found_ratio]
            return f"https://www.zacks.com/stock/chart/{ticker}/fundamental/{ratio_code}"

        if self.lang == 'vi':
            return "Xin lỗi! Không xác định được yêu cầu của bạn"
        return "Sorry! Could not figure out what the user wants"

Overwriting core/financial_ratio_adapter.py


In [72]:
%%writefile utils/pdf_parser.py
import re
from io import StringIO
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfpage import PDFPage


def convert_pdf_to_txt(path: str) -> str:
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)

    with open(path, 'rb') as fp:
        interpreter = PDFPageInterpreter(rsrcmgr, device)
        for page in PDFPage.get_pages(fp, set(), check_extractable=True):
            interpreter.process_page(page)

    text = retstr.getvalue()
    device.close()
    retstr.close()
    return text


def clean_pdf_text(raw_text: str) -> list:
    lines = raw_text.splitlines()
    lines = [l for l in lines if l.strip() not in ('', ' ')]
    cleaned = [re.sub(r"[^a-zA-Z0-9ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠàáâãèéêìíòóôõùúăđĩũơƯĂẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼỀỀỂẾưăạảấầẩẫậắằẳẵặẹẻẽềểếễệỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪễệỉịọỏốồổỗộớờởỡợụủứừỬỮỰỲỴÝỶỸửữựỳỵỷỹ\s]+", " ", l)
               for l in lines]
    return cleaned


def pdf_stats(docs: list) -> dict:
    """Thống kê nhanh số trang/dòng/từ - dùng để in log gọn gàng thay vì
    in cả trăm dòng text ra console."""
    total_words = sum(len(d.split()) for d in docs)
    return {
        "Số dòng văn bản": len(docs),
        "Tổng số từ": total_words,
        "Trung bình từ/dòng": round(total_words / len(docs), 1) if docs else 0,
    }

Overwriting utils/pdf_parser.py


In [73]:
%%writefile utils/visualizer.py
"""
Visualizer utilities - dùng để hiển thị kết quả đẹp hơn trong console
và xuất ra hình ảnh (wordcloud, biểu đồ portfolio, bảng topic).
"""

import os
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS


# ============================================================
# 0. Console formatting helpers - làm log/console đẹp hơn
# ============================================================
def print_header(title: str, char="=", width=70):
    """In tiêu đề section lớn, căn giữa"""
    print("\n" + char * width)
    print(title.center(width))
    print(char * width)


def print_subheader(title: str, char="-", width=60):
    """In tiêu đề nhỏ hơn cho từng bước con"""
    print("\n" + char * width)
    print(f" {title}")
    print(char * width)


def print_kv_table(data: dict, title: str = None, value_fmt="{:.2f}"):
    """In 1 dict dạng bảng key-value căn chỉnh đẹp (VD: portfolio theo ticker)"""
    if title:
        print(f"\n{title}")
    if not data:
        print("  (không có dữ liệu)")
        return
    max_key_len = max(len(str(k)) for k in data.keys())
    for k, v in data.items():
        try:
            v_str = value_fmt.format(v)
        except (ValueError, TypeError):
            v_str = str(v)
        print(f"  {str(k).ljust(max_key_len)} : {v_str}")


def print_topics_table(topics: dict):
    """In danh sách topic LDA dạng bảng dễ đọc thay vì list thô"""
    print("\n  Chủ đề phát hiện được:")
    for topic_name, words in topics.items():
        words_str = ", ".join(words)
        print(f"  - {topic_name}: {words_str}")


# ============================================================
# 1. WordCloud - trực quan hóa từ khóa xuất hiện nhiều trong văn bản
# ============================================================
def plot_wordcloud(text: str, save_path=None, title=None, show=True):
    stopwords = set(STOPWORDS)
    wc = WordCloud(background_color="black", max_words=2000,
                    stopwords=stopwords, width=1200, height=800)
    wc.generate(text)

    plt.figure(figsize=(14, 10))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis("off")
    if title:
        plt.title(title, fontsize=18, color="white", pad=15)
        plt.gcf().patch.set_facecolor('black')

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, facecolor=plt.gcf().get_facecolor(), bbox_inches='tight')
        print(f"[INFO] Đã lưu wordcloud tại: {save_path}")
    if show:
        plt.show()
    else:
        plt.close()


# ============================================================
# 2. Biểu đồ danh mục đầu tư (portfolio value theo ticker)
# ============================================================
def plot_portfolio_by_ticker(results_per_ticker: dict, initial_cash=100000,
                              save_path=None, title="Ket qua giao dich theo ma co phieu",
                              show=True):
    """
    Vẽ biểu đồ cột so sánh Ending Portfolio Value giữa các ticker,
    tô màu xanh (lãi) / đỏ (lỗ) so với vốn ban đầu.
    """
    if not results_per_ticker:
        print("[WARN] Không có dữ liệu portfolio để vẽ.")
        return

    tickers = list(results_per_ticker.keys())
    values = list(results_per_ticker.values())
    colors = ['#2ecc71' if v >= initial_cash else '#e74c3c' for v in values]

    plt.figure(figsize=(12, 6))
    bars = plt.bar(tickers, values, color=colors, edgecolor='black', linewidth=0.5)
    plt.axhline(y=initial_cash, color='gray', linestyle='--', linewidth=1,
                label=f'Von ban dau ({initial_cash:,.0f})')

    for bar, v in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                  f'{v:,.0f}', ha='center', va='bottom', fontsize=9)

    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel("Gia tri danh muc (Portfolio Value)")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
        print(f"[INFO] Đã lưu biểu đồ portfolio tại: {save_path}")
    if show:
        plt.show()
    else:
        plt.close()


def plot_sentiment_distribution(sentiments, save_path=None,
                                 title="Phan bo diem Sentiment", show=True):
    """Histogram phân bố điểm sentiment"""
    plt.figure(figsize=(10, 5))
    plt.hist(sentiments, bins=30, color='#3498db', edgecolor='black', alpha=0.8)
    plt.axvline(x=0, color='red', linestyle='--', linewidth=1, label='Nguong trung lap (0)')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Diem Sentiment")
    plt.ylabel("So luong bai bao")
    plt.legend()
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
        print(f"[INFO] Đã lưu biểu đồ phân bố sentiment tại: {save_path}")
    if show:
        plt.show()
    else:
        plt.close()


def plot_correlation_heatmap(corr_df, save_path=None,
                              title="Tuong quan Sentiment vs Return", show=True):
    """Heatmap tương quan giữa các phương pháp sentiment và eventRet"""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr_df.values, cmap='RdYlGn', vmin=-1, vmax=1)

    n_cols = len(corr_df.columns)
    ax.set_xticks(range(n_cols))
    ax.set_yticks(range(n_cols))
    ax.set_xticklabels(corr_df.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr_df.columns)

    for i in range(n_cols):
        for j in range(n_cols):
            val = corr_df.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        color='black', fontsize=10)

    plt.colorbar(im, ax=ax, label='He so tuong quan')
    plt.title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=120)
        print(f"[INFO] Đã lưu heatmap tương quan tại: {save_path}")
    if show:
        plt.show()
    else:
        plt.close()


# ============================================================
# 3. Topic modeling visualization (pyLDAvis)
# ============================================================
def plot_topic_model(lda_model, feature_matrix, vectorizer):
    try:
        import pyLDAvis
        import pyLDAvis.sklearn
        panel = pyLDAvis.sklearn.prepare(lda_model, feature_matrix, vectorizer)
        pyLDAvis.display(panel)
        return panel
    except Exception as e:
        print(f"[WARN] Không thể vẽ pyLDAvis: {e}")
        return None

Overwriting utils/visualizer.py


In [74]:
%%writefile data/lexicons/viet_lexicon.csv
word,score
tăng_trưởng,1.0
lợi_nhuận,0.8
lãi,0.7
tăng_giá,0.8
tăng_vốn,0.6
tăng_trần,1.2
đột_phá,1.0
kỷ_lục,1.0
khởi_sắc,0.9
phục_hồi,0.7
bứt_phá,1.0
vượt_kế_hoạch,0.9
vượt_dự_báo,0.9
niêm_yết,0.5
cổ_tức,0.6
chia_cổ_tức,0.7
mua_lại,0.5
sáp_nhập,0.4
hợp_tác,0.5
ký_kết,0.4
đầu_tư,0.4
mở_rộng,0.6
tăng_trưởng_doanh_thu,0.9
tăng_trưởng_lợi_nhuận,1.0
lãi_ròng,0.8
lãi_gộp,0.7
biên_lợi_nhuận,0.5
dòng_tiền_dương,0.8
thanh_khoản_tốt,0.7
tín_hiệu_tích_cực,0.8
triển_vọng_tích_cực,0.8
nâng_hạng,0.9
nâng_dự_báo,0.8
khuyến_nghị_mua,0.9
định_giá_hấp_dẫn,0.7
cơ_hội_đầu_tư,0.6
tăng_trưởng_bền_vững,1.0
ổn_định,0.5
an_toàn,0.4
minh_bạch,0.5
hiệu_quả,0.6
thành_công,0.8
thịnh_vượng,0.9
phát_triển,0.6
đi_lên,0.6
hồi_phục,0.7
lạc_quan,0.7
tin_tưởng,0.5
kỳ_vọng,0.4
thua_lỗ,-1.0
phá_sản,-1.5
vỡ_nợ,-1.5
nợ_xấu,-1.2
khủng_hoảng,-1.3
sụt_giảm,-0.9
giảm_giá,-0.7
giảm_sàn,-1.2
lao_dốc,-1.1
sụp_đổ,-1.4
suy_thoái,-1.1
suy_giảm,-0.8
mất_thanh_khoản,-1.0
rút_vốn,-0.7
bán_ròng,-0.6
cắt_lỗ,-0.8
thâm_hụt,-0.8
âm_vốn,-1.0
lỗ_ròng,-0.9
lỗ_lũy_kế,-1.0
cảnh_báo,-0.7
rủi_ro,-0.6
điều_tra,-0.9
khởi_tố,-1.3
truy_tố,-1.3
gian_lận,-1.2
thao_túng,-1.2
lừa_đảo,-1.4
sai_phạm,-1.0
vi_phạm,-0.9
phạt,-0.8
đình_chỉ,-1.0
hủy_niêm_yết,-1.3
giải_thể,-1.2
đóng_cửa,-1.0
sa_thải,-0.9
cắt_giảm,-0.7
nợ_công,-0.8
lạm_phát,-0.7
mất_giá,-0.8
bất_ổn,-0.8
tiêu_cực,-0.7
bi_quan,-0.7
hoảng_loạn,-1.1
tháo_chạy,-1.1
đổ_vỡ,-1.3
thất_bại,-0.9
kiện_tụng,-0.8
tranh_chấp,-0.6
hỗ_trợ,0.4
kết_nối,0.3
hợp_tác,0.4
giải_thưởng,0.7
huy_động,0.3
ưu_đãi,0.5
tài_trợ,0.3
khen_thưởng,0.6
vinh_danh,0.7
đột_phá_công_nghệ,0.8
chuyển_đổi_số,0.5
số_hóa,0.4
mở_rộng_mạng_lưới,0.5
tăng_lãi_suất,-0.3
giảm_lãi_suất,0.3
siết_chặt,-0.5
thanh_tra,-0.6
kiểm_tra,-0.3
xử_phạt,-0.9
đình_chỉ_giao_dịch,-1.1
cảnh_báo_rủi_ro,-0.7

Overwriting data/lexicons/viet_lexicon.csv


In [75]:
%%writefile preprocess_vietnamese_news.py
"""
Tiền xử lý dữ liệu tin tức tài chính tiếng Việt (crawl từ VnExpress).
"""

import os
import re
import pandas as pd
from underthesea import word_tokenize

RAW_PATH = "data/raw/vietnamese_rawdata.csv"
STOPWORDS_PATH = "data/lexicons/vietnamese-stopwords.txt"
OUTPUT_PATH = "data/processed/processed_news.csv"


def load_stopwords(path: str) -> set:
    try:
        with open(path, encoding="utf-8") as f:
            raw_words = [line.strip() for line in f if line.strip()]
        stopwords_set = set(w.replace(" ", "_") for w in raw_words)
        stopwords_set.update(raw_words)
        print(f"[INFO] Đã load {len(raw_words)} stopwords từ {path}")
        return stopwords_set
    except FileNotFoundError:
        print(f"[WARN] Không tìm thấy {path}. Dùng tập stopwords rút gọn mặc định.")
        return {"là", "và", "của", "có", "được", "cho", "này", "đó", "các",
                "một", "những", "trong", "với", "để", "khi", "đã", "sẽ"}


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[\xa0\x92\x93\x94\u200b]", " ", text)
    text = re.sub(
        r"[^0-9a-zàáâãèéêìíòóôõùúăđĩũơưạảấầẩẫậắằẳẵặẹẻẽềềểếễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹ\s]",
        " ", text
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_vietnamese(text: str) -> str:
    if not text:
        return ""
    try:
        return word_tokenize(text, format="text")
    except Exception as e:
        print(f"[WARN] Lỗi tokenize: {e}")
        return text


def remove_stopwords(tokenized_text: str, stopwords: set) -> str:
    tokens = tokenized_text.split()
    return " ".join([t for t in tokens if t not in stopwords])


def preprocess_pipeline(text: str, stopwords: set) -> str:
    cleaned = clean_text(text)
    tokenized = tokenize_vietnamese(cleaned)
    return remove_stopwords(tokenized, stopwords)


def main():
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    print(f"[INFO] Đang đọc dữ liệu từ {RAW_PATH} ...")
    df = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
    print(f"[INFO] Số dòng dữ liệu gốc: {len(df)}")

    df = df.rename(columns={
        "Tiêu đề": "title", "Đường dẫn": "url", "Nội dung chi tiết": "content"
    })
    df = df.dropna(subset=["title", "content"])
    df = df[df["content"].str.strip() != ""]

    before = len(df)
    df = df.drop_duplicates(subset=["url"])
    print(f"[INFO] Đã xóa {before - len(df)} dòng trùng lặp URL")

    stopwords = load_stopwords(STOPWORDS_PATH)

    print("[INFO] Đang tiền xử lý tiêu đề...")
    df["title_clean"] = df["title"].apply(clean_text)
    df["title_processed"] = df["title"].apply(lambda x: preprocess_pipeline(x, stopwords))

    print("[INFO] Đang tiền xử lý nội dung chi tiết...")
    df["content_clean"] = df["content"].apply(clean_text)
    df["content_processed"] = df["content"].apply(lambda x: preprocess_pipeline(x, stopwords))
    df["content_word_count"] = df["content_processed"].apply(lambda x: len(x.split()))

    output_cols = ["title", "url", "title_clean", "title_processed",
                   "content", "content_clean", "content_processed", "content_word_count"]
    df_out = df[output_cols]

    df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print(f"[INFO] Đã lưu {len(df_out)} dòng vào {OUTPUT_PATH}")
    return df_out


if __name__ == "__main__":
    main()

Overwriting preprocess_vietnamese_news.py


In [76]:
%%writefile main.py
"""
Chạy đầy đủ 3 Case Study cho cả pipeline tiếng Anh và tiếng Việt.
Đã tích hợp visualizer + pdf_parser để output đẹp hơn:
- Wordcloud cho văn bản Case Study 3
- Biểu đồ portfolio value theo ticker cho Case Study 1
- Heatmap tương quan sentiment vs return
- Bảng topic LDA được format rõ ràng
Tất cả hình ảnh được lưu vào results/{lang}/...
"""

import os
import pandas as pd

from core.english_pipeline import EnglishBookPipeline
from core.vietnamese_pipeline import VietnameseRealDataPipeline
from utils.pdf_parser import convert_pdf_to_txt, clean_pdf_text, pdf_stats
from utils.visualizer import (
    print_header, print_subheader, print_kv_table, print_topics_table,
    plot_wordcloud, plot_portfolio_by_ticker,
    plot_sentiment_distribution, plot_correlation_heatmap
)

RESULTS_DIR = "results"


def _results_path(lang, filename):
    path = os.path.join(RESULTS_DIR, lang, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    return path


# ================================================================
# ENGLISH PIPELINE
# ================================================================
def run_english_pipeline():
    print_header("ENGLISH PIPELINE")

    eng = EnglishBookPipeline(
        tickers=['AAPL', 'MSFT', 'AMZN', 'GOOG', 'FB',
                 'WMT', 'JPM', 'TSLA', 'NFLX', 'ADBE'],
        start='2010-01-01',
        end='2018-12-31',
        headline_zip_path="data/raw/Raw Headline Data.zip",
        stock_lex_path="data/lexicons/stock_lex.csv"
    )

    # ---- Case Study 1 ----
    print_subheader("Case Study 1: Sentiment Analysis Trading Strategy")
    try:
        use_lstm = os.path.exists("models/saved/lstm_sentiment.h5")
        if not use_lstm and os.path.exists("data/raw/LabelledNewsData.csv"):
            from models.lstm_sentiment import run_lstm_pipeline
            run_lstm_pipeline(csv_path="data/raw/LabelledNewsData.csv", epochs=3)
            use_lstm = True

        result, results_per_ticker = eng.run_case_study_1(use_lstm=use_lstm) \
            if hasattr(eng, 'run_case_study_1') else (None, {})

        print_kv_table(results_per_ticker, title="Kết quả cuối cùng theo mã cổ phiếu:",
                       value_fmt="{:,.2f}")

        plot_portfolio_by_ticker(
            results_per_ticker,
            title="EN - Ending Portfolio Value theo mã cổ phiếu",
            save_path=_results_path("english", "portfolio_by_ticker.png"),
            show=False
        )

        if result is not None and 'sentiment_lex' in result.columns:
            plot_sentiment_distribution(
                result['sentiment_lex'].dropna(),
                title="EN - Phân bố Sentiment (Lexicon-based)",
                save_path=_results_path("english", "sentiment_distribution.png"),
                show=False
            )
            corr_cols = [c for c in ['sentiment_textblob', 'sentiment_lex',
                                      'sentiment_lstm', 'eventRet'] if c in result.columns]
            corr_df = result[corr_cols].corr()
            plot_correlation_heatmap(
                corr_df, title="EN - Tương quan Sentiment vs eventRet",
                save_path=_results_path("english", "correlation_heatmap.png"),
                show=False
            )
    except Exception as e:
        print(f"[ERROR] EN Case Study 1 lỗi: {e}")

    # ---- Case Study 2 ----
    print_subheader("Case Study 2: Chatbot Financial Ratio Assistant")
    try:
        eng.run_case_study_2("What is the ROE for Apple?")
        eng.run_case_study_2("Tell me the PE for Delta?")
        eng.run_case_study_2("Hi, how are you?")
    except Exception as e:
        print(f"[ERROR] EN Case Study 2 lỗi: {e}")

    # ---- Case Study 3 ----
    print_subheader("Case Study 3: Document Summarization (LDA)")
    try:
        raw = convert_pdf_to_txt("data/raw/10K.pdf")
        docs = [d for d in clean_pdf_text(raw) if d.strip() != ""]

        print_kv_table(pdf_stats(docs), title="Thống kê văn bản PDF (10-K):")

        plot_wordcloud(
            " ".join(docs), title="EN - WordCloud báo cáo 10-K",
            save_path=_results_path("english", "wordcloud_10k.png"), show=False
        )

        model, topics, panel = eng.run_case_study_3(docs, n_topics=5, n_top_words=10)
        print_topics_table(topics)

    except FileNotFoundError:
        print("[WARN] Không tìm thấy data/raw/10K.pdf -> bỏ qua EN Case Study 3.")
    except Exception as e:
        print(f"[ERROR] EN Case Study 3 lỗi: {e}")

    return eng


# ================================================================
# VIETNAMESE PIPELINE
# ================================================================
def run_vietnamese_pipeline():
    print_header("VIETNAMESE PIPELINE")

    viet = VietnameseRealDataPipeline(
        tickers=['ACB', 'BID', 'CTG', 'FPT', 'GAS',
                 'HPG', 'MBB', 'NKG', 'REE', 'VCB',
                 'VNM', 'VIC'],   # thêm 2 mã bổ sung, code sẽ tự thử tải
        start='2020-01-01',
        end='2024-12-31',
        news_csv_path="data/processed/processed_news.csv",
        stopwords_path="data/lexicons/vietnamese-stopwords.txt",
        lexicon_path="data/lexicons/viet_lexicon.csv"
    )

    # ---- Case Study 1 ----
    print_subheader("Case Study 1: Sentiment Analysis Trading Strategy")
    try:
        output = viet.run_case_study_1()
        result, results_per_ticker = output if isinstance(output, tuple) else (output, {})

        print_kv_table(results_per_ticker, title="Kết quả cuối cùng theo mã cổ phiếu:",
                       value_fmt="{:,.2f}")
        if results_per_ticker:
            plot_portfolio_by_ticker(
                results_per_ticker, title="VN - Ending Portfolio Value theo ma co phieu",
                save_path=_results_path("vietnamese", "portfolio_by_ticker.png"), show=False
            )
        if result is not None and 'sentiment' in result.columns:
            plot_sentiment_distribution(
                result['sentiment'].dropna(),
                title="VN - Phan bo Sentiment (Lexicon tu xay)",
                save_path=_results_path("vietnamese", "sentiment_distribution.png"), show=False
            )
    except Exception as e:
        print(f"[ERROR] VN Case Study 1 lỗi: {e}")

    # ---- Case Study 2 ----
    print_subheader("Case Study 2: Chatbot tra cứu tỷ số tài chính")
    try:
        viet.run_case_study_2("Cho tôi xem tỷ số ROE của Vinamilk")
        viet.run_case_study_2("Doanh thu của FPT là bao nhiêu")
        viet.run_case_study_2("Xin chào")
    except Exception as e:
        print(f"[ERROR] VN Case Study 2 lỗi: {e}")

    # ---- Case Study 3 ----
    print_subheader("Case Study 3: Tóm tắt tài liệu (LDA)")
    try:
        raw = convert_pdf_to_txt("data/raw/BaoCaoThuongNien.pdf")
        docs = [d for d in clean_pdf_text(raw) if d.strip() != ""]
        source_desc = "PDF Báo cáo thường niên"
    except FileNotFoundError:
        print("[WARN] Không tìm thấy data/raw/BaoCaoThuongNien.pdf.")
        print("       -> Vui lòng tải Annual Report PDF từ trang IR của công ty VN.")
        print("       -> Tạm thời dùng dữ liệu tin tức đã xử lý thay thế.")
        try:
            df = pd.read_csv("data/processed/processed_news.csv", encoding="utf-8-sig")
            docs = df['content'].dropna().tolist()
            source_desc = "Tin tức đã xử lý (fallback)"
        except Exception as e:
            print(f"[ERROR] VN Case Study 3 (fallback) lỗi: {e}")
            docs, source_desc = [], None
    except Exception as e:
        print(f"[ERROR] VN Case Study 3 lỗi: {e}")
        docs, source_desc = [], None

    if docs:
        try:
            print_kv_table(pdf_stats(docs), title=f"Thống kê văn bản ({source_desc}):")

            plot_wordcloud(
                " ".join(docs), title=f"VN - WordCloud ({source_desc})",
                save_path=_results_path("vietnamese", "wordcloud.png"), show=False
            )

            model, topics, panel = viet.run_case_study_3(docs, n_topics=5, n_top_words=10)
            print_topics_table(topics)
        except Exception as e:
            print(f"[ERROR] VN Case Study 3 (visualize) lỗi: {e}")

    return viet


if __name__ == "__main__":
    eng_pipeline = run_english_pipeline()
    viet_pipeline = run_vietnamese_pipeline()

    print_header("HOÀN TẤT CHẠY 3 CASE STUDY CHO CẢ 2 NGÔN NGỮ")
    print(f"📁 Tất cả biểu đồ/wordcloud đã được lưu vào thư mục: {RESULTS_DIR}/")
    print(f"   ├── english/   (portfolio, sentiment, correlation, wordcloud, ...)")
    print(f"   └── vietnamese/")

Overwriting main.py
